In [9]:
import json
import pandas as pd
from pathlib import Path

def json_to_df(path):

    with open(path, 'r') as file:
        data = json.load(file)

    titles = []
    contexts = []
    ids = []
    is_impossibles = []
    questions = []
    answer_texts = []
    answer_starts = []

    L = data['data']
    for i in range(len(L)):
        contract = L[i]
        title = contract['title']
        paragraphs = contract['paragraphs'][0]
        context = paragraphs['context'].strip()
        for item in paragraphs['qas']:
            id = item['id'].strip()
            is_impossible = item['is_impossible']
            question = item['question'].strip()
            questionStart = question.find('"')
            questionEnd = question[questionStart+1:].find('"') + questionStart+1
            question = question[questionStart+1:questionEnd]
            if item['answers'] != []:
                answer_text = item['answers'][0]['text'].strip()
                answer_start = item['answers'][0]['answer_start']
            else:
                answer_text = None
                answer_start = None
            titles.append(title)
            contexts.append(context)
            ids.append(id)
            is_impossibles.append(is_impossible)
            questions.append(question)
            answer_texts.append(answer_text)
            answer_starts.append(answer_start)

    d = {'title': titles,
         'context': contexts,
         'id': ids,
         'is_impossible': is_impossibles,
         'question': questions,
         'answer_text': answer_texts,
         'answer_start': answer_starts,
        }

    return pd.DataFrame(d)

project_root = Path.cwd().parent
df_train = json_to_df(project_root / 'data/train_separate_questions.json')
print(df_train.shape)
df_train.head()

(22450, 7)


,title,context,id,is_impossible,question,answer_text,answer_start
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,Document Name,DISTRIBUTOR AGREEMENT,44.0
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,Parties,Distributor,244.0
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,Parties,Electric City of Illinois L.L.C.,49574.0
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,Parties,Electric City of Illinois LLC,212.0
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,Parties,Company,197.0


In [10]:
df_train_cleaned = df_train.dropna()
df_train_cleaned.shape

(11180, 7)

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

X_train, y_train = df_train_cleaned['answer_text'], df_train_cleaned['question']

text_classifier = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

text_classifier.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U34](41,)","['Affiliate License-Licensee','Affiliate License-Licensor', 'Agreement Date',...,'Unlimited/All-You-Can-Eat-License', 'Volume Restriction','Warranty Duration']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [12]:
project_root = Path.cwd().parent
df_test = json_to_df(project_root / 'data/test.json')
df_test_cleaned = df_test.dropna()
X_test, y_test = df_test_cleaned['answer_text'], df_test_cleaned['question']
predictions = text_classifier.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.60

Classification Report:
                                    precision    recall  f1-score   support

        Affiliate License-Licensee       0.00      0.00      0.00        12
        Affiliate License-Licensor       0.00      0.00      0.00         6
                    Agreement Date       0.67      0.97      0.79        93
                   Anti-Assignment       0.49      0.97      0.65        72
                      Audit Rights       0.55      0.95      0.69        38
                  Cap On Liability       0.39      0.98      0.55        44
                 Change Of Control       1.00      0.04      0.07        26
 Competitive Restriction Exception       0.00      0.00      0.00        16
               Covenant Not To Sue       1.00      0.04      0.08        24
                     Document Name       1.00      0.89      0.94       102
                    Effective Date       0.75      0.09      0.15        70
                       Exclusivity       0.00   

c:\Users\fulle\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\fulle\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\fulle\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave